# ML Distribuito con ML Spark

## 0. Setup dell'ambiente

In [1]:
import os
import sys
import shutil
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# Su Windows: decommenta e adatta se winutils non e' gia' nelle variabili di sistema
# os.environ["HADOOP_HOME"] = r"C:\hadoop"
# os.environ["PATH"] = os.environ["PATH"] + r";C:\hadoop\bin"

print("Interprete    :", sys.executable)
print("HADOOP_HOME   :", os.environ.get("HADOOP_HOME", "NON IMPOSTATO"))
print("SPARK_LOCAL_IP:", os.environ.get("SPARK_LOCAL_IP"))

Interprete    : C:\Users\Allysa\Desktop\MachineLearning-in-Spark\.venv\Scripts\python.exe
HADOOP_HOME   : C:\Users\Allysa\Downloads\hadoop-3.5.0
SPARK_LOCAL_IP: 127.0.0.1


## 1. Avvio della sessione Spark

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("InVehicleCouponRecommendation")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.python.authenticate.socketTimeout", "60")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark  :", spark.version)
print("Shuffle:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark  : 4.2.0
Shuffle: 4


In [3]:
# Smoke test del canale JVM <-> Python worker
assert spark.sparkContext.parallelize([1, 2, 3]).map(lambda x: x * 2).collect() == [2, 4, 6]
print("Test Spark: Perfetto!")

Test Spark: Perfetto!


## 2. Caricamento del dataset

In [6]:
DATA_PATH = Path("data/in-vehicle-coupon-recommendation.csv")
if DATA_PATH.exists():
    print(f"File trovato in \n{DATA_PATH.resolve()}")
else:
    raise ValueError("File non trovato!")

File trovato in 
C:\Users\Allysa\Desktop\MachineLearning-in-Spark\in-vehicle-coupon-recommendation.csv


In [7]:
df = spark.read.csv(str(DATA_PATH), header=True, inferSchema=True)